# USDA Cattle Slaughter Data Processing

This notebook processes the USDA weekly cattle slaughter data from `Dcowslt-13.xlsx` and converts it to more readable formats.

## Data Source
- **File**: `cattle_data/Dcowslt-13.xlsx`
- **Description**: U.S. Federally Inspected Cow Slaughter By Region, Weekly, 1984-Present
- **Units**: 1000 Head
- **Regions**: 10 USDA regions covering the continental United States

## Output Files
1. **cattle_data_clean.csv** - Wide format with all regions and totals
2. **cattle_data_clean.parquet** - Compressed parquet version
3. **cattle_data_long.csv** - Long format for easier analysis and plotting

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Packages loaded successfully")

Packages loaded successfully


## 1. Read Raw Data

The Excel file has complex multi-row headers. We'll read it with proper header handling.

In [2]:
# Read the Excel file - skip first 7 rows to get to data
input_file = Path('../../cattle_data/Dcowslt-13.xlsx')
df_raw = pd.read_excel(input_file, sheet_name='B', header=None, skiprows=8)

print(f"Raw data shape: {df_raw.shape}")
print(f"Date range: {df_raw[0].min()} to {df_raw[0].max()}")
df_raw.head()

Raw data shape: (2254, 28)
Date range: 1984-01-07 00:00:00 to 2027-03-13 00:00:00


,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,1984-01-07,1.0,5.8,6.8,11.9,13.8,10.3,21.3,27.6,40.2,...,4.4,9.1,90.5,180.3,89.8,0.498,NaN,NaN,NaN,NaN
1,1984-01-14,2.0,6.5,6.8,9.7,11.5,9.6,19.1,28.9,39.4,...,4.2,8.8,86.7,163.3,76.6,0.469,NaN,NaN,NaN,NaN
2,1984-01-21,3.0,6.9,7.3,10.9,12.4,9.2,17.9,28.6,40.6,...,3.9,7.9,90.0,169.1,79.1,0.468,NaN,NaN,NaN,NaN
3,1984-01-28,4.0,6.6,6.9,10.7,13.0,9.7,18.2,27.3,37.5,...,4.5,7.9,88.6,159.5,70.9,0.445,NaN,NaN,NaN,NaN
4,1984-02-04,5.0,6.4,6.7,9.9,12.1,8.9,17.2,25.0,34.7,...,4.5,8.3,81.1,150.4,69.3,0.461,NaN,NaN,NaN,NaN


## 2. Create Readable Column Names

Generate descriptive column names based on the header structure:
- Regions 1-10 with dairy and beef+dairy columns
- Reported, calculated, and difference totals

In [3]:
# Define column names based on the header structure
column_names = [
    'date',
    'week',
    # Region 1 & 2 combined
    'region_1_2_dairy',
    'region_1_2_beef_dairy',
    # Region 3
    'region_3_dairy',
    'region_3_beef_dairy',
    # Region 4
    'region_4_dairy',
    'region_4_beef_dairy',
    # Region 5
    'region_5_dairy',
    'region_5_beef_dairy',
    # Region 6
    'region_6_dairy',
    'region_6_beef_dairy',
    # Region 7
    'region_7_dairy',
    'region_7_beef_dairy',
    # Region 8
    'region_8_dairy',
    'region_8_beef_dairy',
    # Region 9
    'region_9_dairy',
    'region_9_beef_dairy',
    # Region 10
    'region_10_dairy',
    'region_10_beef_dairy',
    # Reported totals
    'reported_total_dairy',
    'reported_total_beef_dairy',
    # Calculated totals
    'calculated_total_beef',
    'calculated_pct_beef',
    'calculated_total_dairy',
    'calculated_total_beef_dairy',
    # Difference (reported - calculated)
    'diff_dairy',
    'diff_beef_dairy'
]

# Assign column names
df_raw.columns = column_names

print(f"Columns assigned: {len(column_names)}")
print("\nColumn names:")
for i, col in enumerate(column_names):
    print(f"{i:2d}. {col}")

Columns assigned: 28

Column names:
 0. date
 1. week
 2. region_1_2_dairy
 3. region_1_2_beef_dairy
 4. region_3_dairy
 5. region_3_beef_dairy
 6. region_4_dairy
 7. region_4_beef_dairy
 8. region_5_dairy
 9. region_5_beef_dairy
10. region_6_dairy
11. region_6_beef_dairy
12. region_7_dairy
13. region_7_beef_dairy
14. region_8_dairy
15. region_8_beef_dairy
16. region_9_dairy
17. region_9_beef_dairy
18. region_10_dairy
19. region_10_beef_dairy
20. reported_total_dairy
21. reported_total_beef_dairy
22. calculated_total_beef
23. calculated_pct_beef
24. calculated_total_dairy
25. calculated_total_beef_dairy
26. diff_dairy
27. diff_beef_dairy


## 3. Clean and Validate Data

In [4]:
# Create clean dataframe
df_clean = df_raw.copy()

# Ensure date column is datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])

# Convert week to integer
df_clean['week'] = pd.to_numeric(df_clean['week'], errors='coerce').astype('Int64')

# Convert all numeric columns to float
numeric_cols = [col for col in df_clean.columns if col not in ['date', 'week']]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Remove any rows where date is missing
df_clean = df_clean.dropna(subset=['date'])

# Sort by date
df_clean = df_clean.sort_values('date').reset_index(drop=True)

print(f"Clean data shape: {df_clean.shape}")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Total weeks: {len(df_clean)}")
print(f"Years covered: {df_clean['date'].dt.year.nunique()}")

df_clean.head(10)

Clean data shape: (2254, 28)
Date range: 1984-01-07 00:00:00 to 2027-03-13 00:00:00
Total weeks: 2254
Years covered: 44


,date,week,region_1_2_dairy,region_1_2_beef_dairy,region_3_dairy,region_3_beef_dairy,region_4_dairy,region_4_beef_dairy,region_5_dairy,region_5_beef_dairy,...,region_10_dairy,region_10_beef_dairy,reported_total_dairy,reported_total_beef_dairy,calculated_total_beef,calculated_pct_beef,calculated_total_dairy,calculated_total_beef_dairy,diff_dairy,diff_beef_dairy
0,1984-01-07,1,5.8,6.8,11.9,13.8,10.3,21.3,27.6,40.2,...,4.4,9.1,90.5,180.3,89.8,0.498,NaN,NaN,NaN,NaN
1,1984-01-14,2,6.5,6.8,9.7,11.5,9.6,19.1,28.9,39.4,...,4.2,8.8,86.7,163.3,76.6,0.469,NaN,NaN,NaN,NaN
2,1984-01-21,3,6.9,7.3,10.9,12.4,9.2,17.9,28.6,40.6,...,3.9,7.9,90.0,169.1,79.1,0.468,NaN,NaN,NaN,NaN
3,1984-01-28,4,6.6,6.9,10.7,13.0,9.7,18.2,27.3,37.5,...,4.5,7.9,88.6,159.5,70.9,0.445,NaN,NaN,NaN,NaN
4,1984-02-04,5,6.4,6.7,9.9,12.1,8.9,17.2,25.0,34.7,...,4.5,8.3,81.1,150.4,69.3,0.461,NaN,NaN,NaN,NaN
5,1984-02-11,6,6.2,6.4,9.8,11.9,8.0,17.3,26.2,36.6,...,3.6,7.3,78.5,152.9,74.4,0.487,NaN,NaN,NaN,NaN
6,1984-02-18,7,5.8,6.1,9.4,11.5,8.0,17.8,24.4,34.9,...,3.5,7.3,76.5,146.3,69.8,0.477,NaN,NaN,NaN,NaN
7,1984-02-25,8,5.6,5.8,8.7,10.6,7.3,15.2,24.0,30.4,...,3.8,8.0,72.3,139.1,66.8,0.480,NaN,NaN,NaN,NaN
8,1984-03-03,9,6.2,6.5,6.9,11.4,7.4,18.0,21.9,30.4,...,3.9,8.3,69.1,144.1,75.0,0.520,NaN,NaN,NaN,NaN
9,1984-03-10,10,6.1,6.5,8.6,10.5,7.0,18.4,21.6,31.4,...,3.8,7.7,68.3,142.5,74.2,0.521,NaN,NaN,NaN,NaN


## 4. Data Quality Checks

In [5]:
# Check for missing values
print("Missing values by column:")
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "="*60)

# Summary statistics for key columns
print("\nSummary statistics for reported totals:")
print(df_clean[['reported_total_dairy', 'reported_total_beef_dairy', 'calculated_total_beef']].describe())

Missing values by column:
                             Missing Count  Missing %
week                                    12       0.53
region_1_2_dairy                        61       2.71
region_1_2_beef_dairy                   61       2.71
region_3_dairy                          97       4.30
region_3_beef_dairy                     97       4.30
region_4_dairy                         113       5.01
region_4_beef_dairy                     61       2.71
region_5_dairy                         218       9.67
region_5_beef_dairy                    218       9.67
region_6_dairy                          61       2.71
region_6_beef_dairy                     61       2.71
region_7_dairy                         270      11.98
region_7_beef_dairy                    270      11.98
region_8_dairy                         738      32.74
region_8_beef_dairy                    713      31.63
region_9_dairy                          61       2.71
region_9_beef_dairy                     61       2.71
re

## 5. Export Wide Format Data

Save the cleaned data in wide format (one row per week, all regions as columns).

In [6]:
# Export to CSV
output_csv = Path('../../cattle_data/cattle_data_clean.csv')
df_clean.to_csv(output_csv, index=False)
print(f"Saved to: {output_csv}")
print(f"File size: {output_csv.stat().st_size / 1024:.1f} KB")

# Export to parquet for efficient storage
output_parquet = Path('../../cattle_data/cattle_data_clean.parquet')
df_clean.to_parquet(output_parquet, index=False, compression='snappy')
print(f"\nSaved to: {output_parquet}")
print(f"File size: {output_parquet.stat().st_size / 1024:.1f} KB")
print(f"Compression ratio: {output_csv.stat().st_size / output_parquet.stat().st_size:.1f}x")

Saved to: ../../cattle_data/cattle_data_clean.csv
File size: 366.8 KB

Saved to: ../../cattle_data/cattle_data_clean.parquet
File size: 129.1 KB
Compression ratio: 2.8x


## 6. Create Long Format for Analysis

Reshape the data to long format for easier filtering, grouping, and visualization.

Each row will represent:
- date
- region (1_2, 3, 4, ..., 10)
- category (dairy, beef_dairy)
- value (in 1000 head)

In [7]:
# Extract regional data only (exclude totals for now)
regional_cols = [col for col in df_clean.columns if col.startswith('region_')]
df_regional = df_clean[['date', 'week'] + regional_cols].copy()

# Reshape to long format
df_long = df_regional.melt(
    id_vars=['date', 'week'],
    var_name='region_category',
    value_name='slaughter_1000_head'
)

# Split region_category into region and category
# Match region numbers (including 1_2 for combined regions 1&2)
# Then match either 'beef_dairy' or 'dairy' at the end
df_long[['region', 'category']] = df_long['region_category'].str.extract(r'region_(\d+(?:_\d+)?)_(beef_dairy|dairy)$')

# Clean up
df_long = df_long.drop('region_category', axis=1)
df_long = df_long[['date', 'week', 'region', 'category', 'slaughter_1000_head']]

# Remove rows with missing values
df_long = df_long.dropna(subset=['slaughter_1000_head'])

# Sort by date and region
df_long = df_long.sort_values(['date', 'region', 'category']).reset_index(drop=True)

print(f"Long format shape: {df_long.shape}")
print(f"Unique regions: {df_long['region'].nunique()}")
print(f"Categories: {df_long['category'].unique()}")
print(f"\nFirst 20 rows:")
df_long.head(20)

Long format shape: (37289, 5)
Unique regions: 9
Categories: ['beef_dairy' 'dairy']

First 20 rows:


,date,week,region,category,slaughter_1000_head
0,1984-01-07,1,10,beef_dairy,9.1
1,1984-01-07,1,10,dairy,4.4
2,1984-01-07,1,1_2,beef_dairy,6.8
3,1984-01-07,1,1_2,dairy,5.8
4,1984-01-07,1,3,beef_dairy,13.8
5,1984-01-07,1,3,dairy,11.9
6,1984-01-07,1,4,beef_dairy,21.3
7,1984-01-07,1,4,dairy,10.3
8,1984-01-07,1,5,beef_dairy,40.2
9,1984-01-07,1,5,dairy,27.6


In [11]:
# Export long format to CSV
output_long_csv = Path('../../cattle_data/cattle_data_long.csv')
df_long.to_csv(output_long_csv, index=False)
print(f"Saved to: {output_long_csv}")
print(f"File size: {output_long_csv.stat().st_size / 1024:.1f} KB")

Saved to: ../../cattle_data/cattle_data_long.csv
File size: 1063.3 KB


## 7. Summary Statistics and Visualizations

In [12]:
# Summary by region
print("Average weekly slaughter by region and category (1000 head):")
summary_by_region = df_long.groupby(['region', 'category'])['slaughter_1000_head'].agg(['mean', 'median', 'min', 'max']).round(1)
print(summary_by_region)

print("\n" + "="*60)

# Total by region (sum dairy + beef_dairy)
print("\nAverage weekly total slaughter by region (1000 head):")
total_by_region = df_long.groupby(['region'])['slaughter_1000_head'].sum().sort_values(ascending=False)
print(total_by_region.round(1))

Average weekly slaughter by region and category (1000 head):
                   mean  median  min   max
region category                           
10     beef_dairy   7.2     6.6  1.5  15.9
       dairy        3.6     3.0  0.8   9.8
1_2    beef_dairy   1.6     0.9  0.1  10.0
       dairy        1.4     0.8  0.1   6.9
3      beef_dairy  10.0     9.9  4.4  18.2
       dairy        7.9     7.7  3.7  13.5
4      beef_dairy  13.3    13.0  5.6  29.8
       dairy        4.1     3.6  1.6  14.9
5      beef_dairy  27.6    27.4  0.3  43.5
       dairy       18.2    18.0  9.4  38.2
6      beef_dairy  19.1    19.0  4.6  40.6
       dairy        4.2     3.2  0.7  11.9
7      beef_dairy  17.1    16.0  7.5  43.8
       dairy        2.6     1.8  0.0  13.6
8      beef_dairy   7.0     6.8  0.0  16.5
       dairy        1.5     1.5  0.0   5.6
9      beef_dairy  14.7    14.7  7.1  24.8
       dairy       11.6    12.0  3.4  20.8


Average weekly total slaughter by region (1000 head):
region
5      93334.5
9

In [13]:
# Time series trends - annual averages
df_clean['year'] = df_clean['date'].dt.year
annual_avg = df_clean.groupby('year')[['reported_total_dairy', 'reported_total_beef_dairy', 'calculated_total_beef']].mean().round(1)

print("Annual average weekly slaughter (1000 head):")
print(annual_avg.tail(10))

print("\n" + "="*60)
print("\nOverall trends:")
print(f"First year average: {annual_avg.iloc[0]['reported_total_beef_dairy']:.1f}")
print(f"Last year average: {annual_avg.iloc[-1]['reported_total_beef_dairy']:.1f}")
print(f"Change: {annual_avg.iloc[-1]['reported_total_beef_dairy'] - annual_avg.iloc[0]['reported_total_beef_dairy']:.1f} (1000 head/week)")

Annual average weekly slaughter (1000 head):
      reported_total_dairy  reported_total_beef_dairy  calculated_total_beef
year                                                                        
2018                  60.5                      118.5                   58.0
2019                  61.7                      122.8                   61.2
2020                  58.4                      120.8                   62.4
2021                  59.6                      127.9                   68.3
2022                  58.5                      134.1                   75.6
2023                  59.2                      126.8                   67.6
2024                  52.1                      106.6                   54.5
2025                  50.5                       95.6                   45.1
2026                  52.9                       92.8                   39.8
2027                   NaN                        NaN                    NaN


Overall trends:
First year av

## 8. Data Dictionary

### Wide Format (`cattle_data_clean.csv`)

| Column | Description | Units |
|--------|-------------|-------|
| date | Week ending date | Date |
| week | Week number of year | Integer |
| region_X_dairy | Dairy cow slaughter for region X | 1000 head |
| region_X_beef_dairy | Beef and dairy cow slaughter for region X | 1000 head |
| reported_total_dairy | Reported total dairy slaughter | 1000 head |
| reported_total_beef_dairy | Reported total beef and dairy slaughter | 1000 head |
| calculated_total_beef | Calculated total beef slaughter | 1000 head |
| calculated_pct_beef | Calculated percentage beef | Percent |
| calculated_total_dairy | Calculated total dairy from regions | 1000 head |
| calculated_total_beef_dairy | Calculated total beef+dairy from regions | 1000 head |
| diff_dairy | Difference (reported - calculated) dairy | 1000 head |
| diff_beef_dairy | Difference (reported - calculated) beef+dairy | 1000 head |

### Long Format (`cattle_data_long.csv`)

| Column | Description | Units |
|--------|-------------|-------|
| date | Week ending date | Date |
| week | Week number of year | Integer |
| region | USDA region (1_2, 3, 4, ..., 10) | String |
| category | Slaughter category (dairy or beef_dairy) | String |
| slaughter_1000_head | Number of cattle slaughtered | 1000 head |

### USDA Regions

- **Region 1 & 2**: CT, ME, NH, VT, MA, RI, NY, NJ
- **Region 3**: DE, MD, PA, WV, VA
- **Region 4**: AL, FL, GA, KY, MS, NC, SC, TN
- **Region 5**: IL, IN, MI, MN, OH, WI
- **Region 6**: AR, LA, NM, OK, TX
- **Region 7**: IA, KS, MO, NE
- **Region 8**: CO, MT, ND, SD, UT, WY
- **Region 9**: AZ, CA, HI, NV
- **Region 10**: AK, ID, OR, WA

## Summary

Successfully processed USDA cattle slaughter data:

- **Wide format**: All regions and totals in columns (`cattle_data_clean.csv` and `.parquet`)
- **Long format**: Reshaped for easy analysis (`cattle_data_long.csv`)
- **Data span**: 1984 to present
- **Temporal resolution**: Weekly
- **Spatial coverage**: 10 USDA regions (entire US)

Files are ready for integration with MERRA-2 climate data for livestock heat stress analysis.